# Statistical Analysis for RQ1 and RQ2

This notebook contains the statistical analyses addressing the effects of explanation conditions on trust, system understanding, and mental workload.

The analysis includes assumption checks (Shapiro-Wilk, Q-Q Plots, Sphericity), repeated-measures ANOVAs, Holm-Bonferroni-corrected t-tests, and Friedman tests.

Results are plotted as Boxplots.

In [ ]:
import numpy as np
import pandas as pd
import pingouin as pg

from scipy.stats import shapiro, ttest_rel

from functions import paired_ttests, plot_qq_by_condition, stylized_boxplot

# Data Import

The block-level dataset is created by running `preprocess_score_data.ipynb`.

The dataframe contains one row per experimental block and includes the following variables:

- `participant_id`: unique participant identifier
- `block`: experimental block number
- `route`: route identifier
- `condition`: explanation condition (No explanation, Push explanation, Pull Explanation)
- `trust`: trust score
- `understanding`: system understanding score
- `nasa`: NASA-TLX workload score

In [ ]:
df = pd.read_csv("../DATA/blocks.csv")

In [ ]:
print(df.info())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:


df['condition'] = df['condition'].replace({
    'Nelo': 'No explanation',
    'Coda': 'Push explanation',
    'Lumo': 'Pull explanation'
})

df['explanation'] = df['condition'].replace({
    "No explanation": "No explanation",
    "Push explanation": "Explanation",
    "Pull explanation": "Explanation"
})

df["block"] = df["block"].astype("category")

# Assumption Checks: Shapiro–Wilk Tests and Q-Q Plots for Normality

Normality was assessed using Shapiro–Wilk tests and Q-Q plots separately for each dependent variable and experimental condition.

In [ ]:
for dv in ["trust", "understanding", "nasa"]:
    print(f"\n{'='*50}")
    print(f"{dv.upper()}")
    print(f"{'='*50}")

    for categories in ["condition", "explanation"]:
        for condition in df[categories].unique():
            values = df.loc[df[categories] == condition, dv].dropna()

            stat, p = shapiro(values)

            print(
                f"{condition}: "
                f"W = {stat:.3f}, "
                f"p = {p:.4f}"
            )


In [ ]:
plot_qq_by_condition(df, condition_col="condition", dv="trust", title="Trust")


# Assumption Check: Sphericity

Mauchly's test of sphericity was conducted for repeated-measures ANOVAs to assess whether the sphericity assumption was met.
The test was applied to trust and system understanding, which were analysed using repeated-measures ANOVA.

In [ ]:
for dv in ["trust", "understanding"]:
    print(f"Number of Participants: {df['participant_id'].nunique()}")
    print(f"\n{'='*50}")
    print(f"{dv.upper()}")
    print(f"{'='*50}")

    sphericity = pg.sphericity(
        data=df,
        dv=dv,
        subject="participant_id",
        within="condition"
    )

    print(
        f"Mauchly's test: "
        f"W = {sphericity.W:.3f}, "
        f"χ²({sphericity.dof}) = {sphericity.chi2:.3f}, "
        f"p = {sphericity.pval:.4f}"
    )


# RQ1: Explanations vs No Explanations

In [ ]:
# Paired t-tests one-sided (H1: condition1 >/< condition2)
for dv, alternative in [("trust", "greater"), ("understanding", "greater"), ("nasa", "less")]:
    summary = (
    df
    .groupby(["participant_id", "explanation"])[dv]
    .mean()
    .reset_index()
    )

    pivot = summary.pivot(
        index="participant_id",
        columns="explanation",
        values=dv
    )
    diff = pivot["Explanation"] - pivot["No explanation"]

    t, p = ttest_rel(
        pivot["Explanation"],
        pivot["No explanation"],
        alternative=alternative    # H: explanation greater than no explanation, one-sided test
    )

    print(f"{dv}: t = {t:.3f}, p = {p:.4f}")

    cohens_d = diff.mean() / diff.std(ddof=1)

    print(
        f"Paired t-test: "
        f"t({len(diff)-1}) = {t:.3f}, "
        f"p = {p:.4f}, "
        f"Cohen's d_z = {cohens_d:.3f}"
    )



# RQ2: Repeated-Measures ANOVA

Repeated-measures ANOVAs were conducted for trust and system understanding to test whether explanation condition affected the dependent variables.

Effect sizes are reported as generalized eta squared (η²G).

In [ ]:
for dv in ["trust", "understanding"]:

    aov = pg.rm_anova(
        data=df,
        dv=dv,
        within="condition",
        subject="participant_id",
        detailed=True
        )

    effect = aov.iloc[0]
    error = aov.iloc[1]


    print(
        f"{dv}: "
        f"F({int(effect['DF'])}, {int(error['DF'])}) = "
        f"{effect['F']:.2f}, "
        f"p = {effect['p_unc']:.4f}, "
        f"η²G = {effect['ng2']:.3f}"
    )

# Post-hoc Analysis: Paired-Samples t-tests

Paired-samples t-tests were conducted for pairwise comparisons between explanation conditions.
p-values were corrected for multiple comparisons using the Holm-Bonferroni correction.

In [ ]:
for dv in ["trust", "understanding",]: 
    print(f"\n{'='*50}")
    print(f"{dv.upper()}")
    print(f"{'='*50}")

    results = paired_ttests(df, dv, alternative="greater")

    for pair, r in results.items():

        dz = r["t"] / np.sqrt(r["n"])

        print(
            f"{pair[0]} vs {pair[1]}: "
            f"t({r['n']-1}) = {r['t']:.3f}, "
            f"p_raw = {r['p_raw']:.4f}, "
            f"p_holm = {r['p_holm']:.4f}, "
            f"d_z = {dz:.3f}, "
            f"{r['sig']}"
        )


# Non-parametric Analysis: Friedman Test

Due to violations of the normality assumption, NASA-TLX scores were analysed using a Friedman test as a non-parametric alternative to the repeated-measures ANOVA.

In [ ]:
friedman = pg.friedman(
    data=df,
    dv="nasa",
    within="condition",
    subject="participant_id"
)

result = friedman.iloc[0]

print(
    f"\n{'='*50}"
     "\nNASA\n"
    f"{'='*50}\n"
    f"χ²({int(result['ddof1'])}) = "
    f"{result['Q']:.3f}, "
    f"p = {result['p_unc']:.4f}, "
    f"Kendall's W = {result['W']:.3f}"
)


## Boxplots

In [ ]:
annotations_trust = [
    (("Pull explanation", "No explanation"), "*")]

annotations_understanding = [
    (("Pull explanation", "No explanation"), "***"),
    (("Push explanation", "No explanation"), "***"),
]

stylized_boxplot(df, "understanding", annotations=annotations_understanding)
stylized_boxplot(df, "trust", annotations=annotations_trust)
